In [ ]:
# Cell 1: Import libraries and configure dependencies
# https://www.kaggle.com/jacobramey
# https://github.com/rameyjm7

import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input, Conv1D, MaxPooling1D, Bidirectional, LSTM, Dense, Dropout, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras import mixed_precision
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
import torch
import yaml

# ------------------------------
# Load Noisy Drone RF Signal Classification v2 dataset (local)
# ------------------------------
notebook_dir = Path().resolve()
project_root = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir
cfg_path = project_root / 'configs' / 'local_data_paths.yaml'
if cfg_path.exists():
    local_cfg = yaml.safe_load(cfg_path.read_text())
    dcfg = local_cfg.get('datasets', {}).get('noisy_drone_rf_v2', {})
    path = Path(dcfg.get('data_dir', Path(local_cfg.get('dataset_root', '/scratch/rameyjm7/datasets')) / 'NoisyDroneRFv2'))
else:
    path = Path('/scratch/rameyjm7/datasets/NoisyDroneRFv2')

print('Loading Noisy Drone RF Signal Classification v2 from:', path)
assert path.exists(), f'NoisyDroneRFv2 directory not found: {path}'

sample_re = re.compile(r'IQdata_sample(?P<sample>\d+)_target(?P<target>-?\d+)_snr(?P<snr>-?\d+)\.pt$')
pt_files = sorted(path.rglob('IQdata_sample*_target*_snr*.pt'))
assert pt_files, f'No IQdata_sample*_target*_snr*.pt files found under {path}'
print('PT files found:', len(pt_files))

class_stats_path = path / 'class_stats.csv'
snr_stats_path = path / 'SNR_stats.csv'
if class_stats_path.exists():
    class_stats = pd.read_csv(class_stats_path)
    print('class_stats.csv:')
    display(class_stats.head(20))
else:
    class_stats = None
    print('class_stats.csv not found; using numeric target IDs.')

if snr_stats_path.exists():
    snr_stats = pd.read_csv(snr_stats_path)
    print('SNR_stats.csv:')
    display(snr_stats.head(20))
else:
    snr_stats = None
    print('SNR_stats.csv not found; using SNR values from filenames.')

# ------------------------------
# Load tensors and filename labels
# ------------------------------
def load_pt_iq(filepath):
    obj = torch.load(filepath, map_location='cpu')
    if isinstance(obj, dict):
        for key in ('iq', 'IQ', 'x', 'X', 'data', 'samples'):
            if key in obj:
                obj = obj[key]
                break
    elif isinstance(obj, (tuple, list)):
        obj = obj[0]

    arr = obj.detach().cpu().numpy() if hasattr(obj, 'detach') else np.asarray(obj)
    arr = np.asarray(arr, dtype=np.float32)
    arr = np.squeeze(arr)

    if arr.ndim == 1:
        # Complex vector or interleaved IQ. Preserve complex data as I/Q; otherwise split pairs.
        if np.iscomplexobj(arr):
            arr = np.stack([arr.real, arr.imag], axis=-1).astype(np.float32)
        else:
            assert arr.size % 2 == 0, f'Cannot infer IQ pairs from shape {arr.shape} in {filepath}'
            arr = arr.reshape(-1, 2)
    elif arr.ndim == 2:
        if arr.shape[0] == 2 and arr.shape[1] != 2:
            arr = arr.T
        elif arr.shape[-1] != 2:
            raise ValueError(f'Expected IQ tensor with final dimension 2, got shape {arr.shape} in {filepath}')
    else:
        arr = arr.reshape(arr.shape[0], -1) if arr.shape[-1] != 2 else arr
        if arr.shape[-1] != 2 and arr.shape[0] == 2:
            arr = arr.T
        if arr.shape[-1] != 2:
            raise ValueError(f'Expected IQ tensor with final dimension 2, got shape {arr.shape} in {filepath}')

    return arr.astype(np.float32)

X_list, y_list, snr_list = [], [], []
for f in pt_files:
    m = sample_re.search(f.name)
    if not m:
        continue
    X_list.append(load_pt_iq(f))
    y_list.append(int(m.group('target')))
    snr_list.append(int(m.group('snr')))

min_len = min(x.shape[0] for x in X_list)
X = np.stack([x[:min_len, :2] for x in X_list], axis=0).astype(np.float32)
y_raw = np.asarray(y_list, dtype=np.int64)
snr = np.asarray(snr_list, dtype=np.float32)

classes = np.array(sorted(np.unique(y_raw)), dtype=np.int64)
class_to_index = {c: i for i, c in enumerate(classes)}
y = np.asarray([class_to_index[c] for c in y_raw], dtype=np.int64)
Y = to_categorical(y, num_classes=len(classes))

print('Dataset loaded successfully.')
print('X shape:', X.shape)
print('Y shape:', Y.shape)
print('Classes:', classes.tolist())
print('SNR range:', float(snr.min()), 'to', float(snr.max()))

# ------------------------------
# Train/validation/test split
# ------------------------------
idx = np.arange(X.shape[0])
train_idx, test_idx = train_test_split(idx, test_size=0.20, random_state=1961, stratify=y)
train_idx, val_idx = train_test_split(train_idx, test_size=0.20, random_state=1961, stratify=y[train_idx])

X_train, Y_train, snr_train = X[train_idx], Y[train_idx], snr[train_idx]
X_val,   Y_val,   snr_val   = X[val_idx],   Y[val_idx],   snr[val_idx]
X_test,  Y_test,  snr_test  = X[test_idx],  Y[test_idx],  snr[test_idx]

print(f'Training samples: {X_train.shape[0]} | Validation: {X_val.shape[0]} | Test: {X_test.shape[0]}')

# ------------------------------
# Shuffle
# ------------------------------
np.random.seed(1961)
X_train, Y_train, snr_train = sklearn.utils.shuffle(X_train, Y_train, snr_train, random_state=1961)
X_val,   Y_val,   snr_val   = sklearn.utils.shuffle(X_val, Y_val, snr_val, random_state=1961)
X_test,  Y_test,  snr_test  = sklearn.utils.shuffle(X_test, Y_test, snr_test, random_state=1961)

# ------------------------------
# Normalize IQ per sample
# ------------------------------
def normalize_iq(X):
    Xn = np.empty_like(X)
    for i in range(X.shape[0]):
        scale = np.max(np.abs(X[i])) + 1e-12
        Xn[i] = X[i] / scale
    return Xn

X_train = normalize_iq(X_train)
X_val   = normalize_iq(X_val)
X_test  = normalize_iq(X_test)

# ------------------------------
# Append SNR as a third channel
# ------------------------------
def append_snr_feature(X, snr):
    X_out = []
    snr_scale = max(float(np.max(np.abs(snr))), 1.0)
    for i in range(X.shape[0]):
        snr_col = np.full((X.shape[1], 1), snr[i] / snr_scale)
        X_out.append(np.concatenate([X[i], snr_col], axis=1))
    return np.array(X_out, dtype=np.float32)

X_train = append_snr_feature(X_train, snr_train)
X_val   = append_snr_feature(X_val, snr_val)
X_test  = append_snr_feature(X_test, snr_test)

input_shape = (X_train.shape[1], X_train.shape[2])
num_classes = Y_train.shape[1]

# ------------------------------
# Enable mixed precision
# ------------------------------
mixed_precision.set_global_policy('mixed_float16')

# ------------------------------
# Build CNN + Bidirectional LSTM model
# ------------------------------
def build_cnn_bilstm(input_shape, num_classes):
    model = Sequential([
        Input(shape=input_shape),
        Conv1D(64, 5, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(2),
        Conv1D(128, 3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(2),
        Bidirectional(LSTM(128, return_sequences=True)),
        Dropout(0.3),
        Bidirectional(LSTM(64)),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax', dtype='float32')
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4, clipnorm=1.0),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

model = build_cnn_bilstm(input_shape, num_classes)
model.summary()

# ------------------------------
# Train
# ------------------------------
model_dir = project_root / 'models' / 'noisy_drone_rf_v2'
model_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = model_dir / 'noisy_drone_rf_v2_cnn_bilstm_best.keras'

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True),
    ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
]

history = model.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    epochs=50,
    batch_size=128,
    callbacks=callbacks,
    verbose=1,
)

# ------------------------------
# Evaluate quick test snapshot
# ------------------------------
loss, acc = model.evaluate(X_test, Y_test, verbose=0)
print(f'Noisy Drone RF v2 test accuracy: {acc*100:.2f}%')

Y_pred = model.predict(X_test, verbose=0)
y_true = np.argmax(Y_test, axis=1)
y_pred = np.argmax(Y_pred, axis=1)
print(classification_report(y_true, y_pred, zero_division=0))


In [ ]:
# Cell 2: Set up file paths and runtime configuration
# ------------------------------
# Save final model in .keras format and plot labeled confusion matrices
# ------------------------------
# Define model path
model_path = os.path.join('..', 'models', 'noisy_drone_rf_v2', 'noisy_drone_rf_v2_cnn_bilstm_final.keras')

# Save model architecture, weights, and optimizer state in one file
model.save(model_path, include_optimizer=True)
print(f'Model saved successfully at: {os.path.abspath(model_path)}')


In [ ]:
# Cell 3: Import libraries and configure dependencies
# ================================================
# Noisy Drone RF Signal Classification v2 – Full Evaluation (Standalone Cell)
# ================================================
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import to_categorical
import torch
import yaml

# ---------------------------------------------------------
# Resolve project paths (safe for notebooks anywhere)
# ---------------------------------------------------------
notebook_dir = Path().resolve()
project_root = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir

cfg_path = project_root / 'configs' / 'local_data_paths.yaml'
if cfg_path.exists():
    local_cfg = yaml.safe_load(cfg_path.read_text())
    dcfg = local_cfg.get('datasets', {}).get('noisy_drone_rf_v2', {})
    data_dir = Path(dcfg.get('data_dir', Path(local_cfg.get('dataset_root', '/scratch/rameyjm7/datasets')) / 'NoisyDroneRFv2'))
else:
    data_dir = Path('/scratch/rameyjm7/datasets/NoisyDroneRFv2')
model_path = project_root / 'models' / 'noisy_drone_rf_v2' / 'noisy_drone_rf_v2_cnn_bilstm_final.keras'

print('Notebook directory:', notebook_dir)
print('Project root:', project_root)
print('Dataset directory:', data_dir)
print('Model path:', model_path)

assert data_dir.exists(), f'NoisyDroneRFv2 directory not found: {data_dir}'
assert model_path.exists(), f'Model file missing: {model_path}'

# ---------------------------------------------------------
# Loader helpers
# ---------------------------------------------------------
sample_re = re.compile(r'IQdata_sample(?P<sample>\d+)_target(?P<target>-?\d+)_snr(?P<snr>-?\d+)\.pt$')

def load_pt_iq(filepath):
    obj = torch.load(filepath, map_location='cpu')
    if isinstance(obj, dict):
        for key in ('iq', 'IQ', 'x', 'X', 'data', 'samples'):
            if key in obj:
                obj = obj[key]
                break
    elif isinstance(obj, (tuple, list)):
        obj = obj[0]

    arr = obj.detach().cpu().numpy() if hasattr(obj, 'detach') else np.asarray(obj)
    arr = np.asarray(arr, dtype=np.float32)
    arr = np.squeeze(arr)

    if arr.ndim == 1:
        if np.iscomplexobj(arr):
            arr = np.stack([arr.real, arr.imag], axis=-1).astype(np.float32)
        else:
            assert arr.size % 2 == 0, f'Cannot infer IQ pairs from shape {arr.shape} in {filepath}'
            arr = arr.reshape(-1, 2)
    elif arr.ndim == 2:
        if arr.shape[0] == 2 and arr.shape[1] != 2:
            arr = arr.T
        elif arr.shape[-1] != 2:
            raise ValueError(f'Expected IQ tensor with final dimension 2, got shape {arr.shape} in {filepath}')
    else:
        arr = arr.reshape(arr.shape[0], -1) if arr.shape[-1] != 2 else arr
        if arr.shape[-1] != 2 and arr.shape[0] == 2:
            arr = arr.T
        if arr.shape[-1] != 2:
            raise ValueError(f'Expected IQ tensor with final dimension 2, got shape {arr.shape} in {filepath}')

    return arr.astype(np.float32)

# ---------------------------------------------------------
# Load test data from deterministic split
# ---------------------------------------------------------
pt_files = sorted(data_dir.rglob('IQdata_sample*_target*_snr*.pt'))
assert pt_files, f'No IQdata_sample*_target*_snr*.pt files found under {data_dir}'

X_list, y_list, snr_list = [], [], []
for f in pt_files:
    m = sample_re.search(f.name)
    if not m:
        continue
    X_list.append(load_pt_iq(f))
    y_list.append(int(m.group('target')))
    snr_list.append(int(m.group('snr')))

min_len = min(x.shape[0] for x in X_list)
X = np.stack([x[:min_len, :2] for x in X_list], axis=0).astype(np.float32)
y_raw = np.asarray(y_list, dtype=np.int64)
snr = np.asarray(snr_list, dtype=np.float32)
classes = np.array(sorted(np.unique(y_raw)), dtype=np.int64)
class_to_index = {c: i for i, c in enumerate(classes)}
y = np.asarray([class_to_index[c] for c in y_raw], dtype=np.int64)
Y = to_categorical(y, num_classes=len(classes))

idx = np.arange(X.shape[0])
train_idx, test_idx = train_test_split(idx, test_size=0.20, random_state=1961, stratify=y)

X_test = X[test_idx]
Y_test = Y[test_idx]
snr_test = snr[test_idx]

print('Loaded test shapes:')
print('X_test:', X_test.shape)
print('Y_test:', Y_test.shape)
print('Classes:', classes.tolist())

# ---------------------------------------------------------
# Normalize IQ per sample
# ---------------------------------------------------------
def normalize_iq(X):
    Xn = np.empty_like(X)
    for i in range(X.shape[0]):
        scale = np.max(np.abs(X[i])) + 1e-12
        Xn[i] = X[i] / scale
    return Xn

X_test = normalize_iq(X_test)

# ---------------------------------------------------------
# Append SNR as a 3rd channel
# ---------------------------------------------------------
def append_snr_feature(X, snr):
    out = []
    snr_scale = max(float(np.max(np.abs(snr))), 1.0)
    for i in range(X.shape[0]):
        snr_col = np.full((X.shape[1], 1), snr[i] / snr_scale)
        out.append(np.concatenate([X[i], snr_col], axis=1))
    return np.array(out, dtype=np.float32)

X_test = append_snr_feature(X_test, snr_test)

# ---------------------------------------------------------
# Load model
# ---------------------------------------------------------
print('Loading trained CNN+BiLSTM model...')
model = load_model(model_path)
print('Model loaded.')

# ---------------------------------------------------------
# Evaluate
# ---------------------------------------------------------
loss, acc = model.evaluate(X_test, Y_test, verbose=0)
print(f'\nTest accuracy (all SNRs): {acc*100:.2f}%')

Y_pred = model.predict(X_test, verbose=0)
y_true = np.argmax(Y_test, axis=1)
y_pred = np.argmax(Y_pred, axis=1)

print(classification_report(y_true, y_pred, zero_division=0))

# ---------------------------------------------------------
# Confusion Matrix – All SNRs
# ---------------------------------------------------------
label_names = [f'target_{c}' for c in classes]
cm_all = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(13, 11))
sns.heatmap(
    cm_all, cmap='Blues', annot=False,
    xticklabels=label_names, yticklabels=label_names
)
plt.title('Noisy Drone RF v2 Confusion Matrix – All SNRs')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# Confusion Matrix – High SNR (>= -6 dB)
# ---------------------------------------------------------
high_mask = snr_test >= -6
y_true_high = y_true[high_mask]
y_pred_high = y_pred[high_mask]

cm_high = confusion_matrix(y_true_high, y_pred_high)
plt.figure(figsize=(13, 11))
sns.heatmap(
    cm_high, cmap='Greens', annot=False,
    xticklabels=label_names, yticklabels=label_names
)
plt.title('Noisy Drone RF v2 Confusion Matrix – SNR >= -6 dB')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('\nClassification Report – All SNRs')
print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))

print('\nClassification Report – High SNR (>= -6 dB)')
print(classification_report(y_true_high, y_pred_high, target_names=label_names, zero_division=0))
